Data Gathering <br>
Sources of Data<br>
A vast amount of historical data can be found in files such as:
* MS Word documents
* Emails
* Spreadsheets
* MS PowerPoints
* PDFs
* HTML
* and plaintext files

Public and Private Archives
CSV, JSON, and XML files use plaintext, a common format, and are compatible with a wide range of applications
The Web can be mined for data using a web scraping application
The IoT uses sensors create data
Sensors in smartphones, cars, airplanes, street lamps, and home appliances capture raw data
Open Data and Private Data
1. Open Data
The Open Knowledge Foundation describes Open Data as “any content, information or data that people are free to use, reuse, and redistribute without any legal,
technological, or social restriction.”
1. Private Data
Data related to an expectation of privacy and regulated by a particular country/government
Structured and Unstructured Data
1. Structured Data
Data entered and maintained in fixed fields within a file or record Easily entered, classified, queried, and analyzed Relational databases or spreadsheets
2. Unstructured Data Lacks organization
Raw data Photo contents, audio, video, web pages, blogs, books, journals, white papers, PowerPoint presentations, articles, email, wikis, word processing
documents, and text in general
Example of gathering image data using webcam
Note: Run this snippet using local jupyter notebook

In [2]:
import cv2

# Open webcam
webcam = cv2.VideoCapture(0)

while True:
    check, frame = webcam.read()
    if not check:
        print("Camera not detected.")
        break

    cv2.imshow("Capturing", frame)

    key = cv2.waitKey(1)

    # Press 's' to save image
    if key == ord('s'):
        cv2.imwrite("saved_img.jpg", frame)
        print("Image saved!")

        # Convert to grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Resize to 28x28
        resized = cv2.resize(gray, (28, 28))

        cv2.imwrite("saved_img_final.jpg", resized)
        print("Grayscale 28x28 image saved!")
        break

    # Press 'q' to quit
    elif key == ord('q'):
        print("Camera turned off.")
        break

webcam.release()
cv2.destroyAllWindows()

Camera not detected.


Example of gathering voice data using microphone
Note: Run the snippet of codes using local jupyter notebook

In [3]:
!pip install sounddevice scipy wavio
!pip3 install wavio
!pip3 install scipy
!apt-get install libportaudio2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  libportaudio2
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 65.3 kB of archives.
After this operation, 223 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libportaudio2 amd64 19.6.0-1.1 [65.3 kB]
Fetched 65.3 kB in 1s (106 kB/s)
Selecting previously unselected package libportaudio2:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libportaudio2_19.6.0-1.1_amd64.deb ...
Unpacking libportaudio2:amd64 (19.6.0-1.1) ...
Setting up libportaudio2:amd64 (19.6.0-1.1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.11) ...
/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_ad

In [5]:
import sounddevice as sd
from scipy.io.wavfile import write
import wavio as wv

# Sampling frequency
freq = 44100

# Duration (seconds)
duration = 5

print("Recording...")

# Record
recording = sd.rec(int(duration * freq),
                   samplerate=freq,
                   channels=2)

sd.wait()
print("Recording finished!")

# Save file
write("recording.wav", freq, recording)

# Optional save using wavio
wv.write("recording_wavio.wav", recording, freq, sampwidth=2)

print("Audio saved successfully!")

Recording...


PortAudioError: Error querying device -1

In [9]:
pip install requests bs4

In [8]:
import requests
from bs4 import BeautifulSoup

def getdata(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()  # Raise error if request failed
    return response.text

url = "https://www.google.com/"

try:
    htmldata = getdata(url)
    soup = BeautifulSoup(htmldata, "html.parser")

    # Print image sources
    for img in soup.find_all("img"):
        src = img.get("src")
        if src:
            print(src)

except requests.exceptions.RequestException as e:
    print("Request failed:", e)

/images/branding/googlelogo/1x/googlelogo_white_background_color_272x92dp.png


In [10]:
from requests import get
from bs4 import BeautifulSoup
import pandas as pd

headers = {'Accept-Language': 'en-US,en;q=0.8'}

url = 'https://www.imdb.com/search/title?release_date=2017&sort=num_votes,desc&page=1'
response = get(url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')
movie_containers = soup.find_all('div', class_='lister-item mode-advanced')

names = []
years = []
imdb_ratings = []
metascores = []
votes = []

for container in movie_containers:
    if container.find('div', class_='ratings-metascore') is not None:

        name = container.h3.a.text
        names.append(name)

        year = container.h3.find('span', class_='lister-item-year').text
        years.append(year)

        imdb = float(container.strong.text)
        imdb_ratings.append(imdb)

        m_score = container.find('span', class_='metascore').text.strip()
        metascores.append(int(m_score))

        vote = container.find('span', attrs={'name': 'nv'})['data-value']
        votes.append(int(vote))

df = pd.DataFrame({
    'movie': names,
    'year': years,
    'imdb': imdb_ratings,
    'metascore': metascores,
    'votes': votes
})

print(df.head())

Empty DataFrame
Columns: [movie, year, imdb, metascore, votes]
Index: []


In [12]:

df["year"] = df["year"].astype(str)


df["year"] = (
    df["year"]
        .str.replace(r"\(I{1,3}\)", "", regex=True)
        .str.replace(r"[()]", "", regex=True)
        .str.strip()
        .astype(int)
)

print(df.dtypes)

movie        float64
year           int64
imdb         float64
metascore    float64
votes        float64
dtype: object


In [14]:
df.to_csv("movie_ratings.csv", index=False)
print("CSV file saved successfully!")

CSV file saved successfully!


MULTIPLE PAGE SCRAPING (IMDB)

In [13]:
from time import sleep
import random
import requests
from bs4 import BeautifulSoup
import pandas as pd

pages = [str(i) for i in range(1, 6)]
years_list = ["2017", "2018", "2019", "2020"]

records = []

for yr in years_list:
    for pg in pages:

        url = (
            f"https://www.imdb.com/search/title"
            f"?release_date={yr}&sort=num_votes,desc&page={pg}"
        )

        res = requests.get(url, headers=headers)
        sleep(random.randint(8, 12))

        soup = BeautifulSoup(res.content, "html.parser")
        items = soup.select("div.lister-item.mode-advanced")

        for item in items:
            metascore_block = item.select_one("div.ratings-metascore")
            if metascore_block:

                record = {
                    "movie": item.h3.a.get_text(),
                    "year": item.select_one(".lister-item-year").get_text(),
                    "imdb": float(item.strong.get_text()),
                    "metascore": int(
                        item.select_one(".metascore").get_text(strip=True)
                    ),
                    "votes": int(
                        item.select_one("span[name='nv']")["data-value"]
                    ),
                }

                records.append(record)

movie_ratings = pd.DataFrame(records)

print(movie_ratings.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame
None


# Sumplementary Task

In [15]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
import random

headers = {
    "User-Agent": "Mozilla/5.0"
}

pages = [str(i) for i in range(1, 6)]

titles = []
years = []
genres = []
ratings = []

for page in pages:

    url = f"https://www.imdb.com/search/title/?title_type=feature&release_date=2020-01-01,2020-12-31&page={page}"
    response = requests.get(url, headers=headers)

    sleep(random.randint(5, 8))  # polite delay

    soup = BeautifulSoup(response.text, "html.parser")
    containers = soup.find_all("div", class_="lister-item mode-advanced")

    for container in containers:

        title_tag = container.h3.a
        year_tag = container.find("span", class_="lister-item-year")
        genre_tag = container.find("span", class_="genre")
        rating_tag = container.find("strong")

        titles.append(title_tag.text if title_tag else None)
        years.append(year_tag.text if year_tag else None)
        genres.append(genre_tag.text.strip() if genre_tag else None)
        ratings.append(float(rating_tag.text) if rating_tag else None)

movies_2020 = pd.DataFrame({
    "title": titles,
    "year": years,
    "genre": genres,
    "imdb_rating": ratings
})

print(movies_2020.info())
print(movies_2020.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        0 non-null      float64
 1   year         0 non-null      float64
 2   genre        0 non-null      float64
 3   imdb_rating  0 non-null      float64
dtypes: float64(4)
memory usage: 132.0 bytes
None
Empty DataFrame
Columns: [title, year, genre, imdb_rating]
Index: []


Conclusion:

The activity show how data can be collected from a variety of sources, including documents, sensors, websites, and online databases. It emphasized the need to understand different data types both structured and unstructured and the steps required to process them before analysis. Using tools like Python for web scraping, image capture, and data cleaning made it easier to see how raw data is transformed into useful insights. In essence, gathering data goes beyond simply collecting it; it involves preparing and refining it to ensure it is ready for meaningful analysis and informed decision-making.